# WaterTAP tour — facet edition

A re-implementation of the classic `watertap-1` client walkthrough using the new
**facet-based** query API (`aq.graph()`), on the seawater-RO `model.ttl` in this folder.

Two symmetric moves plus introspection:

| move | meaning | rows |
|------|---------|------|
| `.facets()` | show what predicates/objects are reachable next | (read-only) |
| `.having(step, **φ)` | **stay** on the current nodes, keeping those with such an edge | never multiplies |
| `.follow(step, **φ)` | **move** the cursor to the neighbours along an edge | adds a column |

A **`Profile`** curates the discovery surface (hide noise, name virtual edges). Results come out
with `.count()` / `.nodes()` / `.frame()` / `.select(...)`, and — when the focus nodes are data
points — timeseries via `.data()` / `.dataframe()` / `.latest_data()`. Inspect any selection with
`.to_sparql()`.

## Connect

In [ ]:
import polars as pl
from acquirium import Acquirium
from acquirium.Graframe import P, Profile, Reasoning, like

pl.Config.set_fmt_str_lengths(70)
pl.Config.set_tbl_rows(30)

acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)
# g (the Graframe root) is built in the profile section below.

## Load the model

`insert_graph` reads the file client-side (path relative to this notebook).

In [ ]:
acq.insert_graph("model.ttl", format="turtle", replace=True)
print("graph version:", acq.graph_version())

## Curate the view with a profile

An ontology exposes far more predicates than any one task cares about. A `Profile` shapes the
*discovery surface* — which predicates/types show up in facets — and lets you **name virtual
edges** (property paths) so you traverse `follow("downstream")` instead of
`follow(P(connectedTo).plus())`.

`Profile.base()` hides schema noise (`rdf`/`rdfs`/`owl`/`sh`, class/shape objects); we layer the
water-domain predicates and a few named paths on top. Profiles shape discovery only — you can
still `follow`/`having` a hidden predicate explicitly, or pass `raw=True` to a facet call.

In [ ]:
water = Profile.base().with_(
    # predicates worth seeing (namespace globs + a few exacts):
    allow=["s223:", "nawi:", "qudt:hasQuantityKind", "qudt:hasUnit", "s223:ofSubstance"],
    # ...minus the low-level connection plumbing:
    deny=[
        "s223:cnx", "s223:connected", "s223:connectedThrough", "s223:hasConnectionPoint",
        "s223:isConnectionPointOf", "s223:hasBoundaryConnectionPoint", "s223:connectsAt",
        "s223:connectsThrough", "s223:connectsTo", "s223:connectsFrom", "s223:connectedFrom",
    ],
    # named virtual edges (paths this domain actually cares about):
    edges={
        "downstream": "s223:connectedTo+",                      # transitive flow
        "upstream":   "^s223:connectedTo+",
        "measures":   "s223:hasProperty",                      # equipment to property
        "quantity":   "s223:hasProperty/qudt:hasQuantityKind", # equipment to quantity kind
    },
)

g = acq.graph(profile=water)

## Query by name (no ontology knowledge needed)

You don't have to know the URIs. **Every slot** — the class in `instances(...)`, the predicate in
`follow`/`having`, *and* the object in `value=` — resolves a natural-language string via
Acquirium's embedding matcher: `"pump"`→`nawi:Pump`, `"has property"`→`s223:hasProperty`,
`value="salt"`→`nawi:Constituent-Salt`. The rules are uniform:

- a full URI is used as-is;
- `prefix:local` with a **known** prefix expands as a CURIE;
- a **typo'd** prefix warns and falls back to fuzzy resolution of the local part;
- a colon-less string is treated as natural language.

`like(text, kind=...)` pins the concept kind when a bare word is ambiguous; a number or `Lit(...)`
forces a real literal. `suggest()` previews matches. (Pass `fuzzy=False` to `aq.graph()` to require
exact CURIEs.)

In [ ]:
# Same queries as the rest of this tour, but by name:
print("pumps:", g.instances("pump").count())
print("pump quantity kinds:",
      g.instances("pump").follow("has property").follow("has quantity kind").frame().to_series().to_list())

# Object values resolve by name too — no like() needed for an unambiguous word:
print("salt properties:", g.instances("observable property").having("of substance", value="salt").count())
# ...use like(text, kind=...) only to pin the kind when a bare word is ambiguous:
salt = g.instances("observable property").having("of substance", value=like("salt", "substance"))
print("salt properties (kind-pinned):", salt.count())

# Ambiguous term? preview and choose:
g.suggest("pump", kind="class")

## Find entities by class

`g.instances(cls)` is the seed; it includes subclasses by default (the reasoning profile).

In [ ]:
pumps = g.instances("nawi:Pump")
print("pumps:", pumps.count())
pumps.frame()

In [ ]:
# Raw firehose vs. the profiled view (named virtual edges surface at the top):
pumps.facets(raw=True).show(12)   # everything the ontology exposes
pumps.facets().show()             # curated + named edges (downstream/measures/...)

## Follow relationships

`follow` walks an edge (moves the cursor). Because the profile named
`downstream = s223:connectedTo+`, you traverse it by name — no `P(...)` in sight. Filter the far
end inline with `is_a=` / `value=`.

In [ ]:
# Everything reachable downstream of pump P1, and just the static mixers among them:
print("reachable downstream of P1:", g.nodes("wbs:P1").follow("downstream").count())
g.nodes("wbs:P1").follow("downstream", is_a="nawi:StaticMixer").frame()

## Data nodes (observable properties)

The measurable "data" are `s223:QuantifiableObservableProperty` nodes, attached to equipment via
`s223:hasProperty` (the named `measures` edge) and `observe`d by sensors.

In [ ]:
props = g.instances("s223:QuantifiableObservableProperty")
print("observable properties:", props.count())
props.facets(by="predicate", direction="out").show()

In [ ]:
# The observable properties attached to a pump, via the named "measures" edge:
g.nodes("wbs:P1").follow("measures").frame()

## Filter data nodes

`having` narrows the current set by an edge condition (stays put, never multiplies rows). The
classic `filter_by_quantity_kind` / `filter_by_unit` / `filter_by_substance` become `having(...)`
calls on the property's edges. Object values resolve by name, so you rarely need the exact URI —
and a facet row can be handed straight to `having` (see below).

In [ ]:
# See what's actually available to filter on:
props.facets(by="pred-obj", direction="out", limit=40).to_polars().filter(
    pl.col("predicate").is_in(["qudt:hasQuantityKind", "qudt:hasUnit", "s223:ofSubstance"])
)

In [ ]:
# Facet rows are *actionable*: pick one and hand it straight to having()/follow() —
# the row already carries its predicate, direction, and object, so you never retype a URI.
f = props.facets(by="pred-obj", direction="out", limit=40)
pressure_row = f.row("qudt:hasQuantityKind", key="qk:Pressure")   # or f.row(<index>)
print("pressure points via facet row:", props.having(pressure_row).count())

In [ ]:
# Values resolve by name — no exact CURIEs required:
print("Pressure           :", props.having("has quantity kind", value="pressure").count())
print("unit kg/s          :", props.having("qudt:hasUnit", value="unit:KiloGM-PER-SEC").count())

salt_flow = (props
    .having("of substance", value="salt")
    .having("qudt:hasUnit", value="unit:KiloGM-PER-SEC"))
print("salt mass flow (kg/s):", salt_flow.count())
salt_flow.frame()

## Inspect the query

Every selection compiles to SPARQL — no black box.

In [ ]:
print(salt_flow.to_sparql())

## Systems

Systems are logical groupings of equipment/junctions/subsystems (`s223:hasMember`).

In [ ]:
g.instances("s223:System").frame()

In [ ]:
# Hierarchy: systems that are members of other systems
(g.instances("s223:System").mark("system")
   .follow("s223:hasMember", is_a="s223:System").mark("subsystem")
   .select("system", "subsystem"))

In [ ]:
# Equipment count per system (direct members that are Equipment)
by_system = (g.instances("s223:System").mark("system")
              .follow("s223:hasMember", is_a="s223:Equipment").mark("equipment"))
(by_system.select("system", "equipment")
          .group_by("system")
          .agg(pl.col("equipment").count().alias("equipment_count"))
          .sort("equipment_count", descending=True))

In [ ]:
# Pumps that are members of a specific system
g.nodes("wbs:pretreatment-system").follow("s223:hasMember", is_a="nawi:Pump").frame()

## All pumps and their observed properties

A join built with waypoints: mark the pump, hop out via `measures` to its properties (and their
quantity kind), mark those, then `select` the columns you want.

In [ ]:
(g.instances("nawi:Pump").mark("pump")
   .follow("measures").mark("property")
   .follow("qudt:hasQuantityKind").mark("quantity")
   .select("pump", "property", "quantity"))

## All data-generating entities within a system

System members → their properties → each property's quantity kind. (Desalination system, whose
equipment carry the observable properties directly.)

In [ ]:
(g.nodes("wbs:desalination-system")
   .follow("s223:hasMember", is_a="s223:Equipment").mark("equipment")
   .follow("measures").mark("property")
   .follow("qudt:hasQuantityKind").mark("quantity")
   .select("equipment", "property", "quantity"))

## Pull timeseries

When the focus nodes are data points (they carry `ref:hasExternalReference`), `.data()` fetches
their timeseries as a `DataObject`; `.dataframe()` / `.latest_data()` are convenience wrappers.
Marks become `entity__*` columns, so `.data().by("<mark>")` groups the series by a waypoint, and
series are auto-converted to each point's unit.

> This seawater-RO `model.ttl` is metadata only (no external references), so the frame below is
> empty. Point at a deployment whose driver ingests data (e.g. the WATERTAP compose profile) and
> the *same code* returns one column per point.

In [ ]:
# The pressure points, and their timeseries (wide: one column per point):
pressure = props.having("qudt:hasQuantityKind", value="qk:Pressure")
print("pressure points:", pressure.nodes())
pressure.dataframe(shape="wide")